# Descriptive Statistics

**Notebook 4 of 6.** Between them, `02_data_cleaning.ipynb` and `03_missing_data.ipynb`
made this dataset *trustworthy*: nothing in it lies about its own type, units or
categories, and you know where its gaps are and what filling them would cost.

All of that was repair work. It is only half of an exploratory data analysis, and this
notebook starts the other half: **what does the data actually say?** One column at a time
here, then two at a time in `05_correlation.ipynb`.

By the end you should be able to:

- classify every column as categorical, ordinal or metric, and say why it matters.
- summarise a column with measures of central tendency and dispersion.
- say when the mean misleads and what to use instead.
- read a distribution from a histogram and a boxplot.
- tell a data error apart from a genuine extreme value.

First the libraries this notebook needs.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualisation
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

Then the data. Notice which file it is: `seattle-weather_clean.csv`, the one
`02_data_cleaning.ipynb` wrote, and **not** a version with the gaps filled in.

That is deliberate. In `03_missing_data.ipynb` you imputed onto *copies*, so the missing
values are still here. Pandas will quietly skip them: `.mean()`, `.describe()` and `.corr()`
all ignore missing values rather than returning `NaN`, so everything below works. Where the
count matters, the output says how many rows it used.

In [ ]:
# the cleaned data from 01_data_cleaning
df_weather = pd.read_csv("../data/seattle-weather_clean.csv", parse_dates=["date"])
df_weather.head()

## 1. What kind of variable is each column?

Before computing anything, name what you are working with. The type of a variable decides
which statistics are meaningful, which plot is readable, and which correlation measure
applies. Getting this wrong is how people end up taking the mean of a postcode.

**Categorical** variables take values from a fixed set of labels with no order.
`weather` is categorical: `sun`, `rain`, `fog`, `drizzle`, `snow`.

**Ordinal** variables have labels that *are* ordered, but with no meaningful distance
between them. A survey answer of poor, fair, good is ordinal. This dataset has none.

**Metric** variables are numbers where distances mean something. They split further:

- **discrete** metric variables count things and take whole values, like the number of
  rainy days in a month.
- **continuous** metric variables can in principle take any value in a range.
  `temp_max`, `temp_min`, `wind` and `precipitation` are all continuous.

`date` is its own case. It is ordered, and differences between dates are meaningful, so it
behaves like a metric variable, but it is rarely used directly as one. Usually you derive
something from it, which is exactly what you will do in `05_correlation.ipynb`.

Pandas has an opinion about types too, but it is a *storage* opinion, not a *meaning* one.
It cannot tell a category stored as text from a free-text comment.

In [ ]:
# what pandas thinks each column is
df_weather.dtypes

> **Exercise:**
>
> Fill in the dictionary below, classifying every column of `df_weather` as one of
> `"categorical"`, `"ordinal"`, `"continuous"` or `"discrete"`. Then run the cell after it
> to check how many you agreed with.

<details><summary>
Click here for a hint...
</summary>

Ask yourself two questions about each column:

1. Can I put the values in a meaningful order?
2. Does the *distance* between two values mean anything?

No to both means categorical. Yes then no means ordinal. Yes to both means metric.
</details>

In [ ]:
# classify each column
variable_types = {
    "date": "continuous",      # ordered with meaningful differences, but usually derived from
    "precipitation": "continuous",
    "temp_max": "continuous",
    "temp_min": "continuous",
    "wind": "continuous",
    "weather": "categorical",
}

This split is worth keeping, because almost everything below is applied to one group or
the other.

In [ ]:
# split the column names into two groups we will reuse
numerical_features = ["precipitation", "temp_max", "temp_min", "wind"]
categorical_features = ["weather"]

print("numerical  :", numerical_features)
print("categorical:", categorical_features)

## 2. Central tendency and dispersion

Two questions describe any numerical column: **where is it centred**, and **how spread out
is it**.

Three measures of centre:

- the **mean**, the arithmetic average. Sensitive to extreme values.
- the **median**, the middle value when sorted. Unmoved by extreme values.
- the **mode**, the most frequent value. The only one of the three that works on
  categorical data.

Four measures of spread:

- the **variance**, the mean squared distance from the mean.
- the **standard deviation**, its square root, back in the original units.
- the **range**, maximum minus minimum. Decided entirely by the two most extreme rows.
- the **interquartile range (IQR)**, the distance between the 25th and 75th percentiles.
  It describes the middle half of the data and ignores the tails.

`.describe()` gives you most of this in one call.

In [ ]:
# summary statistics for every numerical column
df_weather[numerical_features].describe()

Look at `precipitation`. The mean and the median are telling you different stories, and the
difference is not a rounding error.

In [ ]:
# the mean and median disagree sharply for precipitation
print(f"precipitation mean   : {df_weather.precipitation.mean():.2f}")
print(f"precipitation median : {df_weather.precipitation.median():.2f}")
print(f"precipitation skew   : {df_weather.precipitation.skew():.2f}")

The median is `0.0`: on more than half of all days in Seattle it does not rain at all. The
mean is pulled well above zero by a minority of very wet days. **When the mean and the
median disagree, the distribution is skewed**, and the mean on its own will mislead you.

This is not a flaw in the data. It is a fact about rainfall, and it is the sort of thing an
EDA is *for*.

> **Exercise:**
>
> Describe the `wind` column by hand rather than with `.describe()`. Compute its variance,
> standard deviation, range and IQR, and print them. Then compute the mode of the
> categorical `weather` column.

<details><summary>
Click here for a hint...
</summary>

`.var()`, `.std()`, `.min()`, `.max()` and `.quantile()` all exist on a pandas Series.
The IQR is `.quantile(0.75) - .quantile(0.25)`. For the mode, look at `.mode()` or at
`.value_counts()`.
</details>

In [ ]:
# variance, standard deviation, range and IQR of wind
print(f"wind variance : {df_weather.wind.var():.3f}")
print(f"wind std      : {df_weather.wind.std():.3f}")
print(f"wind range    : {df_weather.wind.max() - df_weather.wind.min():.3f}")

iqr = df_weather.wind.quantile(0.75) - df_weather.wind.quantile(0.25)
print(f"wind IQR      : {iqr:.3f}")

# the most common weather category
print(f"\nmost common weather: {df_weather.weather.mode()[0]}")
df_weather.weather.value_counts()

Notice that the standard deviation is the square root of the variance, and that it is
roughly twice the IQR. Neither is wrong. They measure spread in different ways, and the
standard deviation is the one that notices the tails.

## 3. Distributions and outliers

Summary numbers hide shape. Two columns can share a mean and a standard deviation and look
nothing alike. Plot them.

- a **histogram** buckets the values and counts how many land in each bucket.
- a **boxplot** draws the median, the IQR as a box, and anything unusually far outside it
  as individual points.

In [ ]:
# the shape of the two temperature columns
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df_weather.temp_max.hist(ax=axes[0], bins=25, color="#33a5ff")
axes[0].set_title("temp_max")
df_weather.temp_min.hist(ax=axes[1], bins=25, color="#03ac13")
axes[1].set_title("temp_min")
plt.tight_layout();

Both are roughly bell shaped, with a single peak. Now compare that to `precipitation`.

In [ ]:
# a very different shape
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df_weather.precipitation.hist(ax=axes[0], bins=40, color="#33a5ff")
axes[0].set_title("precipitation, histogram")
sns.boxplot(x=df_weather.precipitation, ax=axes[1], color="#33a5ff")
axes[1].set_title("precipitation, boxplot")
plt.tight_layout();

This is the skew you measured in section 2, seen directly. A tall spike at zero, then a long thin tail to the
right. The boxplot marks dozens of days as points beyond the whiskers.

**Those points are not errors.** They are the days it rained hard. This is the distinction
that matters most when hunting outliers:

- an **error** is a value that cannot be true, like the `$` you removed in `02_data_cleaning.ipynb`, or a
  temperature of 500 degrees.
- an **extreme value** is a true value that happens to be far from the rest.

Deleting the second kind because a rule flagged it is one of the most common ways to ruin a
dataset. The rule finds candidates. You decide.

The usual rule is **Tukey's fence**: a value is flagged if it lies more than `1.5 * IQR`
below the first quartile or above the third.

> **Exercise:**
>
> Write a function `iqr_outliers(series)` that returns the rows of a Series flagged by
> Tukey's fence. Apply it to `precipitation` and to `wind`, and print how many each has.
>
> Then answer in a comment: for each column, do you think these are errors or genuine
> extreme values, and what would you do about them?

<details><summary>
Click here for a hint...
</summary>

```python
q1 = series.quantile(0.25)
q3 = series.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
```

Then return the values of the series outside `lower` and `upper`. A boolean mask with `|`
between the two conditions will do it.
</details>

In [ ]:
def iqr_outliers(series):
    """Return the values of `series` that lie beyond 1.5 * IQR from the quartiles.

    Args:
        series: a numerical pandas Series, missing values are ignored.

    Returns:
        A Series holding only the flagged values.
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]


for column in ["precipitation", "wind"]:
    flagged = iqr_outliers(df_weather[column])
    share = len(flagged) / df_weather[column].notna().sum() * 100
    print(f"{column:14} {len(flagged):4} flagged ({share:.1f} % of non-missing)")

# precipitation: genuine extreme values. Seattle has heavy rain days, and the median of 0
#   makes the IQR tiny, so the fence flags anything but a dry day. Keep them, and prefer
#   the median over the mean when summarising this column.
# wind: also genuine. A handful of storm days. Keep them, but check the very largest
#   against the date to make sure they are not a repeated sensor fault.

> **Note.** Tukey's fence assumes a roughly symmetric distribution. On a column as skewed as
> `precipitation` it flags far too much, which is why the boxplot looks the way it does.
> A rule is a starting point, never a verdict.

## 4. The distribution of a categorical column

A histogram buckets numbers. A categorical column has no numbers to bucket, so its
distribution is just the count of each category, and the plot for it is a **bar plot**.

This is the one-variable case of a rule that runs through the next notebook: **the type of
the variable decides the plot.** Reach for a histogram or a boxplot when you have numbers,
a bar plot when you have labels.

In [ ]:
# the distribution of a categorical column is a count per category
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# counts, ordered from most to least common
sns.countplot(data=df_weather, x="weather",
              order=df_weather.weather.value_counts().index, ax=axes[0])
axes[0].set_title("weather, counts")

# the same thing as proportions, which is easier to compare across datasets
proportions = df_weather.weather.value_counts(normalize=True)
sns.barplot(x=proportions.index, y=proportions.values, ax=axes[1])
axes[1].set_title("weather, proportions")
axes[1].set_ylabel("share of days")

plt.tight_layout();

Two things this makes obvious that `.value_counts()` alone does not. Seattle is sunny on
roughly half of all days, which is not the reputation. And `snow` is so rare that any
statistic computed on the snow rows alone will be unreliable: there are only 23 of them.

> **Note.** Order the bars by frequency rather than alphabetically, as above with `order=`. An
> alphabetical axis hides the shape of the distribution, which is the only reason you drew
> it.

## Wrapping up

You can now describe any single column: name its type, find its centre, measure its spread,
see its shape, and tell a data error apart from a value that is merely extreme.

- **numbers** get a histogram or a boxplot, and the mean, median, standard deviation and IQR.
- **categories** get a bar plot, and counts, proportions and the mode.
- when the **mean and the median disagree**, the distribution is skewed and the mean alone
  will mislead you. `precipitation` has a mean of `2.97` and a median of `0.00`.
- an **outlier rule produces candidates, not verdicts.** Tukey's fence flagged 14 % of
  `precipitation`, and almost all of it is real Seattle rain.

Every one of those describes a column **on its own**. The questions that actually get asked
of a dataset are usually about pairs of columns: does it rain harder when the wind is up, is
the weather different in summer, which of these columns would help me predict that one.

Next: `05_correlation.ipynb` takes the columns two at a time.